In [1]:
import pandas as pd
from pathlib import Path

base = Path("../data/processed")
ventes = pd.read_parquet(base / "online_retail_II_cleaned.parquet")
clients = pd.read_parquet(base / "customer_rfm_clusters.parquet")

NOMS = {0: "VIP et très fidèles", 1: "Inactifs et occasionnels",
        2: "Récents à potentiel", 3: "Intermédiaires à risque"}
clients["Segment"] = clients["Cluster"].map(NOMS)
ventes = ventes.merge(clients[["Customer ID", "Segment"]], on="Customer ID", how="left")

kpi = {
    "Chiffre d'affaires": ventes["Montant"].sum(),
    "Nombre de factures": ventes["Invoice"].nunique(),
    "Nombre de clients": ventes["Customer ID"].nunique(),
    "Quantité vendue": ventes["Quantity"].sum(),
}
kpi["Panier moyen"] = kpi["Chiffre d'affaires"] / kpi["Nombre de factures"]
display(pd.Series(kpi).round(2))

ventes["Année"] = ventes["InvoiceDate"].dt.year
display(ventes.groupby("Année")["Montant"].sum().round(2))
display(ventes.groupby("Country")["Montant"].sum().sort_values(ascending=False).head(5).round(2))
display(ventes.groupby("Segment")["Montant"].sum().round(2))
print("Ventes sans segment :", ventes["Segment"].isna().sum())

Chiffre d'affaires    17250156.23
Nombre de factures       36859.00
Nombre de clients         5877.00
Quantité vendue       10508717.00
Panier moyen               468.00
dtype: float64

Année
2009     680601.01
2010    8326143.01
2011    8243412.20
Name: Montant, dtype: float64

Country
United Kingdom    14379924.62
EIRE                616570.54
Netherlands         549970.09
Germany             386490.51
France              324368.96
Name: Montant, dtype: float64

Segment
Inactifs et occasionnels      627644.24
Intermédiaires à risque      2879620.02
Récents à potentiel          1052647.48
VIP et très fidèles         12690244.49
Name: Montant, dtype: float64

Ventes sans segment : 0


In [2]:
paniers = ventes.groupby("Invoice")["Montant"].sum()
print("Panier médian :", round(paniers.median(), 2))

mois = ventes["InvoiceDate"].dt.month
display(ventes[mois <= 11].groupby("Année")["Montant"].sum().round(2))

Panier médian : 302.46


Année
2010    7760150.28
2011    7728981.76
Name: Montant, dtype: float64

Le deuxième résultat te donne le chiffre d’affaires de janvier à novembre pour 2010 et 2011 : c’est la seule comparaison honnête entre les deux années.